# 2.3 ter — Méthode par dictionnaire et règles, DIC-v4 (DIC-v3 + lexique MeSH)

**Point de départ : DIC-v3** (notebook 2.3b), reconstruite ici sans changement et vérifiée à l'identique.

**Pourquoi MeSH ?** Sur VALIDATION, DIC-v3 manque 1 941 mentions, dont environ 1 600 ont une forme jamais vue dans
TRAIN : un dictionnaire construit sur TRAIN ne peut pas les trouver. Les outils de référence de BioCreative V
(tmChem, DNorm) s'appuient au contraire sur de grands lexiques. L'enseignant a validé l'ajout d'un lexique externe.

**MeSH** (*Medical Subject Headings*, NLM) est le vocabulaire médical qui sert aussi à normaliser les annotations de
BC5CDR. On reprend, dans la version 2026 :
- les **descripteurs** des arbres C (maladies) et F03 (troubles mentaux) → Disease, D (produits chimiques et
  médicaments) → Chemical ;
- les **concepts supplémentaires** de substances (→ Chemical) et de maladies rares (→ Disease) ;
- tous leurs synonymes ; un terme écrit à l'envers (« Infarction, Myocardial ») est aussi ajouté à l'endroit.

**Comment l'ajouter sans tout dégrader** (tout est appris sur TRAIN) :

| # | Choix | Raison |
|---|---|---|
| 1 | **Priorité à TRAIN** : une forme annotée dans TRAIN garde la décision de DIC-v3 | TRAIN reflète les conventions d'annotation de BC5CDR |
| 2 | **Filtre par branche** : une branche de MeSH dont les termes sont annotés moins de 30 % du temps dans TRAIN est écartée | BC5CDR n'annote pas les protéines, enzymes ou classes de médicaments (*insulin, enzyme, analgesic*) |
| 3 | **Filtre par terme** : un terme MeSH trouvé dans TRAIN mais annoté moins de 30 % du temps est retiré | même seuil que DIC-v3 (*lead, water…*) |

Les règles de DIC-v3 s'appliquent ensuite sur le dictionnaire élargi. Plusieurs variantes sont comparées sur
VALIDATION ; TEST ne sert à rien construire.

In [1]:
import sys, json, time
from collections import Counter
from pathlib import Path

import pandas as pd

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE))

from tals import erreurs
from src.cdr import charger_jsonl, mentions_reference
from src.extraction import tableau_scores, ecrire_predictions, lire_predictions
from src.dictionnaire_v3 import DictionnaireV3, CONFIG_V3
from src.dictionnaire_v4 import DictionnaireV4
from src.mesh import charger_lexique

RESULTATS = RACINE / "resultats" / "extraction"

documents = charger_jsonl(RACINE / "data" / "corpus_extraction.jsonl")  # écrit par 2.2
train = [d for d in documents if d.partition == "train"]
validation = [d for d in documents if d.partition == "validation"]
test = [d for d in documents if d.partition == "test"]
ref_val = mentions_reference(validation)
textes_val = {d.document_id: d.texte for d in validation}
print(len(train), len(validation), len(test), "documents ;", len(ref_val), "mentions de référence en VALIDATION")

500 500 500 documents ; 9591 mentions de référence en VALIDATION


## DIC-v3 reconstruite

In [2]:
pred_v3 = DictionnaireV3().construire(train).extraire_corpus(validation)
attendu = lire_predictions(RESULTATS / "predictions_dictionnaire_v3_validation.jsonl")
assert pred_v3 == attendu, "DIC-v3 n'est pas reproduite"
print("DIC-v3 reproduite à l'identique :", len(pred_v3), "prédictions sur VALIDATION")

DIC-v3 reproduite à l'identique : 7962 prédictions sur VALIDATION


## Le lexique MeSH

Téléchargé une fois depuis le site de la NLM (≈ 64 Mo, dans `data/mesh/`, non versionné), puis réduit à une liste
(terme, catégorie, partie de MeSH, branches). Les branches sont les numéros d'arbre coupés au 2e niveau :
« D12.776 » = protéines, « D12.125 » = acides aminés, « C10.228 » = maladies du système nerveux central.

In [3]:
lexique = charger_lexique(RACINE / "data" / "mesh")
effectifs = pd.Series(Counter((s, c) for _, c, s, _ in lexique)).rename("termes").rename_axis(["partie", "catégorie"])
print(len(lexique), "termes")
display(effectifs.to_frame())
pd.DataFrame([x for x in lexique if x[0] in ("Infarction, Myocardial", "Myocardial Infarction", "Insulin", "Levodopa",
                                                "remifentanil", "Torsades de Pointes")],
             columns=["terme", "catégorie", "partie", "branches"])

841445 termes


termes
partie         catégorie        
supplementaire Chemical   666305
descripteur    Chemical    95061
               Disease     60672
supplementaire Disease     19407

,terme,catégorie,partie,branches
0,"Infarction, Myocardial",Disease,descripteur,"C14.280,C14.907,C23.550"
1,Insulin,Chemical,descripteur,"D06.472,D12.644"
2,Levodopa,Chemical,descripteur,"D02.092,D02.455,D12.125"
3,Myocardial Infarction,Disease,descripteur,"C14.280,C14.907,C23.550"
4,Torsades de Pointes,Disease,descripteur,"C14.280,C23.550"


## Filtre par branche (appris sur TRAIN)

Tous les termes MeSH sont cherchés dans les textes de TRAIN ; pour chaque branche, on compte la part des occurrences
qui sont annotées avec le bon type et les bonnes bornes. Une branche vue au moins 5 fois et annotée moins de 30 %
du temps est écartée, avec tous ses termes.

In [4]:
dic_v4 = DictionnaireV4(lexique, niveau_branches=2).construire(train)
branches = pd.DataFrame([{"branche": b, "occurrences TRAIN": t, "annotées": a, "taux": round(a / t, 2),
                          "écartée": b in dic_v4.branches_ecartees}
                         for b, (t, a) in dic_v4.branches_stats.items()])
print(len(dic_v4.branches_ecartees), "branches écartées sur", len(branches), "rencontrées dans TRAIN")
branches.sort_values("occurrences TRAIN", ascending=False).head(25)

29 branches écartées sur 275 rencontrées dans TRAIN


,branche,occurrences TRAIN,annotées,taux,écartée
5,D03.383,828,818,0.99,False
1,D03.633,806,801,0.99,False
16,C23.888,799,752,0.94,False
6,D02.455,765,750,0.98,False
7,D02.092,699,689,0.99,False
11,C23.550,514,401,0.78,False
2,D03.132,493,488,0.99,False
4,D27.505,481,83,0.17,True
9,D12.776,468,53,0.11,True
15,C10.597,430,405,0.94,False


## Variantes comparées sur VALIDATION

Chaque variante part de DIC-v3 et n'en change que le dictionnaire (règles identiques).

In [5]:
VARIANTES = {
    "MeSH sans filtre": dict(niveau_branches=0, filtre_termes=False),
    "MeSH + filtre par terme": dict(niveau_branches=0),
    "+ filtre par branche (niveau 1, « D12 »)": dict(niveau_branches=1),
    "+ filtre par branche (niveau 2, « D12.776 »)": dict(niveau_branches=2),
    "niveau 2, descripteurs seuls": dict(niveau_branches=2, sources=("descripteur",)),
}

def ligne(nom, pred, dic=None):
    df = tableau_scores(ref_val, pred, "dictionnaire", "validation").set_index("categorie")
    e = {k: len(v) for k, v in erreurs(ref_val, pred).items()}
    return {"variante": nom, "P": df.loc["global (micro)", "precision"], "R": df.loc["global (micro)", "rappel"],
            "F1": df.loc["global (micro)", "F1"], "F1 Chemical": df.loc["Chemical", "F1"], "F1 Disease": df.loc["Disease", "F1"],
            "termes MeSH ajoutés": len(dic.mesh_ajoutees) if dic else 0, **e}

dics, preds = {}, {"DIC-v3": pred_v3}
for nom, options in VARIANTES.items():
    dics[nom] = DictionnaireV4(lexique, **options).construire(train)
    preds[nom] = dics[nom].extraire_corpus(validation)
effet = pd.DataFrame([ligne(n, p, dics.get(n)) for n, p in preds.items()])
effet.to_csv(RESULTATS / "dictionnaire_v4_effet_validation.csv", index=False)
effet.round(4)

,variante,P,R,F1,F1 Chemical,F1 Disease,termes MeSH ajoutés,correcte,frontiere,type,manquee,parasite
0,DIC-v3,0.8811,0.7314,0.7993,0.8333,0.7569,0,7015,627,8,1941,312
1,MeSH sans filtre,0.7733,0.8598,0.8142,0.8054,0.8268,824542,8246,643,4,698,1771
2,MeSH + filtre par terme,0.8580,0.8633,0.8607,0.8740,0.8426,823925,8280,506,4,801,860
3,"+ filtre par branche (niveau 1, « D12 »)",0.8823,0.8627,0.8724,0.8954,0.8421,692858,8274,465,10,842,629
4,"+ filtre par branche (niveau 2, « D12.776 »)",0.9032,0.8669,0.8847,0.9174,0.8424,443740,8314,437,8,832,446
5,"niveau 2, descripteurs seuls",0.9017,0.8458,0.8729,0.8984,0.8405,93209,8112,462,8,1009,414


## Choix de DIC-v4

On garde la variante qui a le meilleur F1 VALIDATION.

In [6]:
choix = effet.iloc[1:].set_index("variante")["F1"].idxmax()
print("Variante retenue :", choix)
OPTIONS_V4 = VARIANTES[choix]

methode_v4 = DictionnaireV4(lexique, **OPTIONS_V4).construire(train)
t0 = time.perf_counter()
pred_val_v4 = methode_v4.extraire_corpus(validation)
t_val = time.perf_counter() - t0
assert pred_val_v4 == preds[choix]

scores_v4 = tableau_scores(ref_val, pred_val_v4, "dictionnaire_v4", "validation", t_val)
scores_v4.to_csv(RESULTATS / "scores_dictionnaire_v4_validation.csv", index=False)
ecrire_predictions(pred_val_v4, "dictionnaire_v4", validation, RESULTATS / "predictions_dictionnaire_v4_validation.jsonl")
pd.concat([tableau_scores(ref_val, pred_v3, "DIC-v3", "validation"), scores_v4], ignore_index=True)

Variante retenue : + filtre par branche (niveau 2, « D12.776 »)


,methode,partition,categorie,VP,FP,FN,precision,rappel,F1,denominateur_nul,temps_s
0,DIC-v3,validation,Chemical,4055,330,1292,0.9247,0.7584,0.8333,False,None
1,DIC-v3,validation,Disease,2960,617,1284,0.8275,0.6975,0.7569,False,None
2,DIC-v3,validation,global (micro),7015,947,2576,0.8811,0.7314,0.7993,False,None
3,dictionnaire_v4,validation,Chemical,4862,391,485,0.9256,0.9093,0.9174,False,NaN
4,dictionnaire_v4,validation,Disease,3452,500,792,0.8735,0.8134,0.8424,False,NaN
5,dictionnaire_v4,validation,global (micro),8314,891,1277,0.9032,0.8669,0.8847,False,2.79


In [7]:
# Ce que MeSH corrige et ce qu'il introduit par rapport à DIC-v3
def s(m):
    return textes_val[m.doc][m.debut:m.fin]

print("ajouts justes :", len((pred_val_v4 - pred_v3) & ref_val), Counter(s(m) for m in (pred_val_v4 - pred_v3) & ref_val).most_common(15))
print("ajouts faux   :", len((pred_val_v4 - pred_v3) - ref_val), Counter(s(m) for m in (pred_val_v4 - pred_v3) - ref_val).most_common(15))
print("pertes        :", len((pred_v3 - pred_val_v4) & ref_val), Counter(s(m) for m in (pred_v3 - pred_val_v4) & ref_val).most_common(10))

ajouts justes : 1345 [('iohexol', 14), ('remifentanil', 12), ('CB 3717', 11), ('ammonia', 11), ('cibenzoline', 11), ('OCT', 10), ('optic neuropathy', 10), ('suprofen', 10), ('uric acid', 10), ('vitamin C', 10), ('torsade de pointes', 10), ('cis-RA', 9), ('amisulpride', 9), ('P', 9), ('SM-5887', 9)]
ajouts faux   : 270 [('gellan', 8), ('PTH', 8), ('glucagon', 5), ('DOC', 5), ('surface antigen', 5), ('tic', 5), ('PVC', 4), ('essential hypertension', 4), ('neoantigens', 3), ('stenosis', 3), ('placebos', 3), ('mineral', 3), ('chronic urticaria', 3), ('OSCS', 3), ('allergic rhinitis', 3)]
pertes        : 46 [('seizures', 4), ('hypertension', 4), ('iron', 3), ('urticaria', 3), ('seizure', 2), ('sodium', 2), ('pain', 2), ('myocardial infarction', 2), ('myopathy', 2), ('cancer', 2)]


## Gel et prédictions sur TEST

Dictionnaire, lexique filtré et règles sont construits sur TRAIN (choix de la variante sur VALIDATION), puis
appliqués aux textes de TEST ; le score TEST sera calculé dans `5_evaluation.ipynb`.

In [8]:
t0 = time.perf_counter()
pred_test_v4 = methode_v4.extraire_corpus(test)
t_test = time.perf_counter() - t0
ecrire_predictions(pred_test_v4, "dictionnaire_v4", test, RESULTATS / "predictions_dictionnaire_v4_test.jsonl")

gel = {"version": "DIC-v4", "base": "DIC-v3 (notebook 2.3b), reproduite à l'identique", "config": vars(CONFIG_V3),
       "lexique": "MeSH 2026 (NLM), descripteurs C, F03, D + concepts supplémentaires classes 1 et 3",
       "variante": choix, "options": {k: list(v) if isinstance(v, tuple) else v for k, v in OPTIONS_V4.items()},
       "termes_mesh_ajoutes": len(methode_v4.mesh_ajoutees), "termes_mesh_retires": len(methode_v4.mesh_retirees),
       "branches_ecartees": sorted(methode_v4.branches_ecartees), "entrees": len(methode_v4.dic.entrees),
       "temps_s": {"extraction_validation": round(t_val, 2), "extraction_test": round(t_test, 2)},
       "predictions_test": len(pred_test_v4)}
(RESULTATS / "dictionnaire_v4_config_gelee.json").write_text(json.dumps(gel, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(gel, ensure_ascii=False, indent=2))

{
  "version": "DIC-v4",
  "base": "DIC-v3 (notebook 2.3b), reproduite à l'identique",
  "config": {
    "casse": "insensible",
    "seuil_annotation": 0.3,
    "regex": true
  },
  "lexique": "MeSH 2026 (NLM), descripteurs C, F03, D + concepts supplémentaires classes 1 et 3",
  "variante": "+ filtre par branche (niveau 2, « D12.776 »)",
  "options": {
    "niveau_branches": 2
  },
  "termes_mesh_ajoutes": 443740,
  "termes_mesh_retires": 148,
  "branches_ecartees": [
    "C26",
    "C26.844",
    "D01.045",
    "D01.496",
    "D01.786",
    "D01.875",
    "D05.500",
    "D06.347",
    "D08.244",
    "D08.811",
    "D10",
    "D10.627",
    "D12.776",
    "D13.444",
    "D20.280",
    "D23.063",
    "D23.101",
    "D23.529",
    "D26",
    "D26.255",
    "D26.776",
    "D27.505",
    "D27.720",
    "D27.888",
    "E01.370",
    "E05.041",
    "G07.100",
    "N06.230",
    "N06.850"
  ],
  "entrees": 448043,
  "temps_s": {
    "extraction_validation": 2.79,
    "extraction_test": 2.78
 

Pour l'application de démonstration : on enregistre les termes MeSH retenus, pour reconstruire DIC-v4 sans relire
MeSH (DIC-v3 + ces termes), et on vérifie qu'on retrouve exactement les prédictions TEST.

In [9]:
from src.dictionnaire_v4 import verifier_dic_v4

methode_v4.sauver_ajouts(RESULTATS / "dictionnaire_v4_termes_mesh.tsv.gz")
print("DIC-v4 reconstruite à l'identique à partir du fichier :", verifier_dic_v4(RACINE), "prédictions TEST")

DIC-v4 reconstruite à l'identique à partir du fichier : 9550 prédictions TEST
